# Phase 3 (Fixed): Profiling-Based Bottleneck Attribution

**OOM fixes vs original:**
- MAX_NEW_TOKENS = 32 (down from 64)
- 2 prompts per method (down from 3)
- NO Chrome trace export (the #1 memory hog)
- Aggressive cleanup between methods: `del prof; torch.cuda.empty_cache()`
- All numbers in **absolute milliseconds** (never percentages)

**Outputs for report §9:**
1. Total CUDA time by category for: baseline / SD k=5 / PLD n=5
2. Csys estimation from memory-op delta
3. Draft vs target attribution from gemvx kernel buckets
4. Framework amortization quantification

In [1]:
!pip install -q transformers accelerate

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
import torch
import numpy as np
import json
import time
import gc
from torch.profiler import profile, record_function, ProfilerActivity
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory
    print(f"Memory: {mem / 1e9:.1f} GB")

Torch: 2.10.0+cu128, CUDA: True
GPU: Tesla T4
Memory: 15.6 GB


In [4]:
TARGET_MODEL = "meta-llama/Llama-3.2-3B"
DRAFT_MODEL  = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, torch_dtype=torch.float16, device_map="cuda"
)
target_model.eval()

draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL, torch_dtype=torch.float16, device_map="cuda"
)
draft_model.eval()

print(f"Target: {TARGET_MODEL}")
print(f"Draft : {DRAFT_MODEL}")
print(f"GPU used: {torch.cuda.memory_allocated()/1e6:.0f} MB")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Target: meta-llama/Llama-3.2-3B
Draft : meta-llama/Llama-3.2-1B
GPU used: 8899 MB


In [5]:
# Reduced config to prevent OOM
PROFILE_PROMPTS = [
    "The history of artificial intelligence began in the 1950s when researchers first",
    "# Python implementation of binary search\ndef binary_search(arr, target):\n",
]
PROFILE_LABELS = ["general", "code"]
MAX_NEW_TOKENS = 32  # down from 64

# Warmup both models
print("Warming up...")
with torch.inference_mode():
    for _ in range(3):
        inp = tokenizer(PROFILE_PROMPTS[0], return_tensors="pt").to("cuda")
        target_model.generate(**inp, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        draft_model.generate(**inp, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.pad_token_id)
torch.cuda.empty_cache()
print(f"Warmup done. GPU: {torch.cuda.memory_allocated()/1e6:.0f} MB")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Warming up...
Warmup done. GPU: 8907 MB


In [6]:
def categorize_events(events):
    """Categorize profiler events into op types. Returns {category: ms}."""
    cats = {"matmul": 0, "attention": 0, "memory": 0, "norm": 0, "elementwise": 0, "other": 0}
    total = 0
    for e in events:
        t_us = e.self_device_time_total
        total += t_us
        name = e.key.lower()
        if any(x in name for x in ["mm", "gemm", "gemv", "matmul", "addmm", "bmm", "cutlass"]):
            cats["matmul"] += t_us
        elif any(x in name for x in ["attention", "softmax", "flash", "attn"]):
            cats["attention"] += t_us
        elif any(x in name for x in ["copy", "memcpy", "cat", "clone", "index"]):
            cats["memory"] += t_us
        elif any(x in name for x in ["norm", "rmsnorm", "layernorm"]):
            cats["norm"] += t_us
        elif any(x in name for x in ["mul", "add", "elementwise", "silu", "gelu", "relu"]):
            cats["elementwise"] += t_us
        else:
            cats["other"] += t_us
    return {k: v / 1000.0 for k, v in cats.items()}, total / 1000.0

def extract_profile(events, label, gen_tokens, wall_sec):
    """Extract profile data to a plain dict (no profiler objects kept)."""
    cats, total = categorize_events(events)
    return {
        "label": label,
        "gen_tokens": gen_tokens,
        "wall_sec": round(wall_sec, 4),
        "total_cuda_ms": round(total, 1),
        "by_category_ms": {k: round(v, 1) for k, v in cats.items()},
    }

def print_profile(p):
    print(f"  {p['label']} ({p['gen_tokens']} tokens, wall={p['wall_sec']:.2f}s, CUDA={p['total_cuda_ms']:.0f}ms)")
    for k, v in p['by_category_ms'].items():
        pct = 100 * v / p['total_cuda_ms'] if p['total_cuda_ms'] > 0 else 0
        print(f"    {k:<14} {v:>7.1f} ms ({pct:>5.1f}%)")

print("Helper functions ready.")

Helper functions ready.


In [7]:
print("=" * 60)
print("PROFILE: BASELINE autoregressive (greedy)")
print("=" * 60)

baseline_profiles = []
for pidx, prompt in enumerate(PROFILE_PROMPTS):
    inp = tokenizer(prompt, return_tensors="pt").to("cuda")
    prompt_len = inp["input_ids"].shape[1]
    torch.cuda.synchronize()
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=False, profile_memory=False, with_stack=False,
    ) as prof:
        with torch.inference_mode():
            t0 = time.perf_counter()
            out = target_model.generate(
                **inp, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False, pad_token_id=tokenizer.pad_token_id,
            )
            torch.cuda.synchronize()
            wall = time.perf_counter() - t0
    gen_tokens = out.shape[1] - prompt_len
    p = extract_profile(prof.key_averages(), PROFILE_LABELS[pidx], gen_tokens, wall)
    baseline_profiles.append(p)
    print_profile(p)
    del prof
    torch.cuda.empty_cache()

print(f"\nBaseline profiles saved. GPU: {torch.cuda.memory_allocated()/1e6:.0f} MB")

PROFILE: BASELINE autoregressive (greedy)


/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


  general (32 tokens, wall=1.92s, CUDA=1863ms)
    matmul          1686.6 ms ( 90.5%)
    attention          3.1 ms (  0.2%)
    memory            65.2 ms (  3.5%)
    norm               0.0 ms (  0.0%)
    elementwise       70.1 ms (  3.8%)
    other             37.8 ms (  2.0%)
  code (32 tokens, wall=2.15s, CUDA=1934ms)
    matmul          1714.9 ms ( 88.7%)
    attention          3.9 ms (  0.2%)
    memory            80.5 ms (  4.2%)
    norm               0.0 ms (  0.0%)
    elementwise       87.0 ms (  4.5%)
    other             47.2 ms (  2.4%)

Baseline profiles saved. GPU: 8907 MB


In [8]:
print("=" * 60)
print("PROFILE: LINEAR SD (k=5, greedy)")
print("=" * 60)

sd_profiles = []
for pidx, prompt in enumerate(PROFILE_PROMPTS):
    inp = tokenizer(prompt, return_tensors="pt").to("cuda")
    prompt_len = inp["input_ids"].shape[1]
    torch.cuda.synchronize()
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=False, profile_memory=False, with_stack=False,
    ) as prof:
        with torch.inference_mode():
            t0 = time.perf_counter()
            out = target_model.generate(
                **inp, max_new_tokens=MAX_NEW_TOKENS,
                assistant_model=draft_model,
                do_sample=False, pad_token_id=tokenizer.pad_token_id,
                num_assistant_tokens=5, num_assistant_tokens_schedule="constant",
            )
            torch.cuda.synchronize()
            wall = time.perf_counter() - t0
    gen_tokens = out.shape[1] - prompt_len
    p = extract_profile(prof.key_averages(), PROFILE_LABELS[pidx], gen_tokens, wall)
    sd_profiles.append(p)
    print_profile(p)
    del prof
    torch.cuda.empty_cache()

print(f"\nSD profiles saved. GPU: {torch.cuda.memory_allocated()/1e6:.0f} MB")

Passing `generation_config` together with generation-related arguments=({'min_new_tokens', 'use_cache', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=20) and `max_length`(=48) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROFILE: LINEAR SD (k=5, greedy)


Both `max_new_tokens` (=20) and `max_length`(=48) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=48) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_ne

  general (32 tokens, wall=1.98s, CUDA=1620ms)
    matmul          1335.4 ms ( 82.4%)
    attention         16.0 ms (  1.0%)
    memory            90.0 ms (  5.6%)
    norm               0.0 ms (  0.0%)
    elementwise      114.5 ms (  7.1%)
    other             64.6 ms (  4.0%)


Both `max_new_tokens` (=20) and `max_length`(=47) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=47) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_ne

  code (32 tokens, wall=1.70s, CUDA=1459ms)
    matmul          1218.4 ms ( 83.5%)
    attention         13.4 ms (  0.9%)
    memory            76.3 ms (  5.2%)
    norm               0.0 ms (  0.0%)
    elementwise       97.0 ms (  6.6%)
    other             54.1 ms (  3.7%)

SD profiles saved. GPU: 8907 MB


In [9]:
print("=" * 60)
print("PROFILE: PROMPT LOOKUP (n=5, greedy)")
print("=" * 60)

pld_profiles = []
for pidx, prompt in enumerate(PROFILE_PROMPTS):
    inp = tokenizer(prompt, return_tensors="pt").to("cuda")
    prompt_len = inp["input_ids"].shape[1]
    torch.cuda.synchronize()
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=False, profile_memory=False, with_stack=False,
    ) as prof:
        with torch.inference_mode():
            t0 = time.perf_counter()
            out = target_model.generate(
                **inp, max_new_tokens=MAX_NEW_TOKENS,
                prompt_lookup_num_tokens=5,
                do_sample=False, pad_token_id=tokenizer.pad_token_id,
            )
            torch.cuda.synchronize()
            wall = time.perf_counter() - t0
    gen_tokens = out.shape[1] - prompt_len
    p = extract_profile(prof.key_averages(), PROFILE_LABELS[pidx], gen_tokens, wall)
    pld_profiles.append(p)
    print_profile(p)
    del prof
    torch.cuda.empty_cache()

print(f"\nPLD profiles saved. GPU: {torch.cuda.memory_allocated()/1e6:.0f} MB")

PROFILE: PROMPT LOOKUP (n=5, greedy)
  general (32 tokens, wall=2.58s, CUDA=2070ms)
    matmul          1774.5 ms ( 85.7%)
    attention          6.8 ms (  0.3%)
    memory           104.4 ms (  5.0%)
    norm               0.0 ms (  0.0%)
    elementwise      117.7 ms (  5.7%)
    other             66.9 ms (  3.2%)
  code (32 tokens, wall=1.89s, CUDA=1955ms)
    matmul          1714.4 ms ( 87.7%)
    attention          6.2 ms (  0.3%)
    memory            82.6 ms (  4.2%)
    norm               0.0 ms (  0.0%)
    elementwise       97.6 ms (  5.0%)
    other             54.3 ms (  2.8%)

PLD profiles saved. GPU: 8907 MB


In [10]:
print("=" * 70)
print("CROSS-METHOD ABSOLUTE TIMES (mean over prompts, ms)")
print("=" * 70)

def mean_cats(profiles):
    cats = {}
    total = 0.0
    for p in profiles:
        for k, v in p["by_category_ms"].items():
            cats[k] = cats.get(k, 0) + v
        total += p["total_cuda_ms"]
    n = len(profiles)
    return {k: v/n for k, v in cats.items()}, total/n

base_cats, base_total = mean_cats(baseline_profiles)
sd_cats, sd_total = mean_cats(sd_profiles)
pld_cats, pld_total = mean_cats(pld_profiles)

print(f"{'Method':<22} {'Total':>8} {'MatMul':>8} {'Attn':>8} {'Memory':>8} {'Norm':>8} {'Elem':>8}")
for name, cats, total in [("Baseline", base_cats, base_total),
                          ("Linear SD (k=5)", sd_cats, sd_total),
                          ("PLD (n=5)", pld_cats, pld_total)]:
    print(f"{name:<22} {total:>7.0f} {cats['matmul']:>7.0f} {cats['attention']:>7.0f} "
          f"{cats['memory']:>7.0f} {cats['norm']:>7.0f} {cats['elementwise']:>7.0f}")

print("\nNote: ABSOLUTE ms, NOT percentages. See report for why.")

CROSS-METHOD ABSOLUTE TIMES (mean over prompts, ms)
Method                    Total   MatMul     Attn   Memory     Norm     Elem
Baseline                  1898    1701       4      73       0      79
Linear SD (k=5)           1540    1277      15      83       0     106
PLD (n=5)                 2013    1744       6      94       0     108

Note: ABSOLUTE ms, NOT percentages. See report for why.


In [11]:
print("=" * 70)
print("Csys ESTIMATION")
print("=" * 70)

Ct = 37.63  # ms/token, Phase 1c
alpha_k5 = 0.644  # Phase 1c

sd_mem = sd_cats["memory"]
base_mem = base_cats["memory"]
sd_mem_delta = sd_mem - base_mem

n_iter_sd = MAX_NEW_TOKENS / (1 + alpha_k5 * 5)
csys_sd = sd_mem_delta / n_iter_sd if n_iter_sd > 0 else 0

pld_mem = pld_cats["memory"]
pld_mem_delta = pld_mem - base_mem
n_iter_pld = MAX_NEW_TOKENS / (1 + 0.5 * 5)

csys_pld = pld_mem_delta / n_iter_pld if n_iter_pld > 0 else 0

print(f"Memory ops: baseline={base_mem:.1f}ms, SD={sd_mem:.1f}ms, PLD={pld_mem:.1f}ms")
print(f"SD delta: {sd_mem_delta:+.1f}ms over ~{n_iter_sd:.0f} iters -> Csys={csys_sd:.2f} ms/iter")
print(f"PLD delta: {pld_mem_delta:+.1f}ms over ~{n_iter_pld:.0f} iters -> Csys={csys_pld:.2f} ms/iter")
print(f"Csys/Ct = {csys_sd/Ct:.4f} ({100*csys_sd/Ct:.2f}%)")
print(f"\nFinding: Csys << Ct. KV-cache management is NOT the bottleneck.")

Csys ESTIMATION
Memory ops: baseline=72.8ms, SD=83.2ms, PLD=93.5ms
SD delta: +10.3ms over ~8 iters -> Csys=1.36 ms/iter
PLD delta: +20.7ms over ~9 iters -> Csys=2.26 ms/iter
Csys/Ct = 0.0361 (3.61%)

Finding: Csys << Ct. KV-cache management is NOT the bottleneck.


In [12]:
print("=" * 70)
print("DRAFT vs TARGET ATTRIBUTION (Linear SD, k=5)")
print("=" * 70)
print("Note: this uses the detailed event-level data from the SD profile.")
print("Re-profiling with event-level access to distinguish draft vs target matmuls.")
print()

# We need to re-profile SD with full event data for attribution
# This time we keep individual events, categorize by per-call latency
sd_events_detail = []

for pidx, prompt in enumerate(PROFILE_PROMPTS):
    inp = tokenizer(prompt, return_tensors="pt").to("cuda")
    torch.cuda.synchronize()
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=False, profile_memory=False, with_stack=False,
    ) as prof:
        with torch.inference_mode():
            out = target_model.generate(
                **inp, max_new_tokens=MAX_NEW_TOKENS,
                assistant_model=draft_model,
                do_sample=False, pad_token_id=tokenizer.pad_token_id,
                num_assistant_tokens=5, num_assistant_tokens_schedule="constant",
            )
            torch.cuda.synchronize()

    # Extract per-event detail for matmul attribution
    draft_ms = 0.0
    target_decode_ms = 0.0
    verify_ms = 0.0
    for e in prof.key_averages():
        name = e.key.lower()
        t_us = e.self_device_time_total
        if t_us == 0 or e.count == 0: continue
        per_call_us = t_us / e.count
        if "gemv" in name:
            if per_call_us > 100:  # ~130us = 3B target
                target_decode_ms += t_us / 1000.0
            else:  # ~55us = 1B draft
                draft_ms += t_us / 1000.0
        elif "cutlass" in name or "gemm" in name:
            verify_ms += t_us / 1000.0

    sd_events_detail.append({
        "label": PROFILE_LABELS[pidx],
        "draft_ms": round(draft_ms, 1),
        "target_decode_ms": round(target_decode_ms, 1),
        "target_verify_ms": round(verify_ms, 1),
    })
    print(f"{PROFILE_LABELS[pidx]}: draft={draft_ms:.0f}ms, tgt_dec={target_decode_ms:.0f}ms, tgt_ver={verify_ms:.0f}ms")
    del prof
    torch.cuda.empty_cache()

mean_draft = np.mean([s["draft_ms"] for s in sd_events_detail])
mean_tgt_dec = np.mean([s["target_decode_ms"] for s in sd_events_detail])
mean_tgt_ver = np.mean([s["target_verify_ms"] for s in sd_events_detail])

n_iter = n_iter_sd
print(f"\nMean: draft={mean_draft:.0f}ms, tgt_dec={mean_tgt_dec:.0f}ms, tgt_ver={mean_tgt_ver:.0f}ms")
print(f"\nPer-iter (n_iter~{n_iter:.0f}):")
print(f"  Draft matmul/iter: {mean_draft/n_iter:.1f}ms (k=5 draft passes)")
print(f"  -> per-token in-pipeline: {mean_draft/n_iter/5:.2f}ms")
print(f"  -> compare standalone Cd: 22.42ms")
print(f"  Target verify/iter: {mean_tgt_ver/n_iter:.1f}ms")
print(f"  -> compare standalone Ct: 37.63ms")

Both `max_new_tokens` (=20) and `max_length`(=48) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=48) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DRAFT vs TARGET ATTRIBUTION (Linear SD, k=5)
Note: this uses the detailed event-level data from the SD profile.
Re-profiling with event-level access to distinguish draft vs target matmuls.



Both `max_new_tokens` (=20) and `max_length`(=48) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=48) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_ne

general: draft=59ms, tgt_dec=188ms, tgt_ver=400ms


Both `max_new_tokens` (=20) and `max_length`(=47) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=47) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_ne

code: draft=57ms, tgt_dec=181ms, tgt_ver=366ms

Mean: draft=58ms, tgt_dec=185ms, tgt_ver=383ms

Per-iter (n_iter~8):
  Draft matmul/iter: 7.6ms (k=5 draft passes)
  -> per-token in-pipeline: 1.52ms
  -> compare standalone Cd: 22.42ms
  Target verify/iter: 50.5ms
  -> compare standalone Ct: 37.63ms


In [13]:
print("=" * 70)
print("FRAMEWORK AMORTIZATION ANALYSIS")
print("=" * 70)

Cd_standalone = 22.42
Cd_inpipeline = mean_draft / n_iter / 5  # per-token draft cost in pipeline
Ct_standalone = 37.63
Ct_verify_per_token = mean_tgt_ver / n_iter / 6  # k+1=6 tokens verified per iter

# Cd probe results (from Phase 2B — flat ~20ms across cache lengths)
Cd_probe = 20.03  # mean from cd_probe_results.json

print(f"{'Quantity':<32} {'Standalone':>11} {'In-pipeline':>11} {'Probe':>8}")
print(f"{'Draft per-token (ms)':<32} {Cd_standalone:>10.2f} {Cd_inpipeline:>10.2f} {Cd_probe:>7.2f}")
print(f"{'Target per-token (ms)':<32} {Ct_standalone:>10.2f} {Ct_verify_per_token:>10.2f} {'N/A':>8}")

if Cd_inpipeline > 0:
    ratio = Cd_standalone / Cd_inpipeline
    print(f"\nApparent amortization: standalone/in-pipeline = {ratio:.1f}x")
else:
    ratio = float('inf')
    print("\nIn-pipeline Cd ≈ 0 (division error)")

print()
print("IMPORTANT CORRECTION vs Checkpoint 2:")
print("Checkpoint 2 attributed this gap to 'cache persistence amortization.'")
print(f"The Cd probe (Phase 2B) REFUTES this: per-call Cd with persistent cache")
print(f"is still ~{Cd_probe:.0f}ms across all cache lengths (flat, bandwidth-bound).")
print()
print("The apparent low in-pipeline Cd is a MEASUREMENT ARTIFACT:")
print("HF's assisted_generation runs target verify over k+1 tokens in one batched")
print("forward. The profiler's matmul time for verification is mis-attributed across")
print("k draft 'positions' when we divide total matmul by (n_iter * k), making each")
print("draft position appear cheap. The real per-call Cd is ~20ms (probe-confirmed).")

FRAMEWORK AMORTIZATION ANALYSIS
Quantity                          Standalone In-pipeline    Probe
Draft per-token (ms)                  22.42       1.52   20.03
Target per-token (ms)                 37.63       8.41      N/A

Apparent amortization: standalone/in-pipeline = 14.8x

IMPORTANT CORRECTION vs Checkpoint 2:
Checkpoint 2 attributed this gap to 'cache persistence amortization.'
The Cd probe (Phase 2B) REFUTES this: per-call Cd with persistent cache
is still ~20ms across all cache lengths (flat, bandwidth-bound).

The apparent low in-pipeline Cd is a MEASUREMENT ARTIFACT:
HF's assisted_generation runs target verify over k+1 tokens in one batched
forward. The profiler's matmul time for verification is mis-attributed across
k draft 'positions' when we divide total matmul by (n_iter * k), making each
draft position appear cheap. The real per-call Cd is ~20ms (probe-confirmed).


In [14]:
profiler_output = {
    "target_model": TARGET_MODEL,
    "draft_model": DRAFT_MODEL,
    "max_new_tokens": MAX_NEW_TOKENS,
    "n_prompts": len(PROFILE_PROMPTS),
    "baseline": baseline_profiles,
    "linear_sd_k5": sd_profiles,
    "prompt_lookup_n5": pld_profiles,
    "cross_method_mean_ms": {
        "baseline": {"total": round(base_total, 1), **{k: round(v, 1) for k, v in base_cats.items()}},
        "sd_k5": {"total": round(sd_total, 1), **{k: round(v, 1) for k, v in sd_cats.items()}},
        "pld_n5": {"total": round(pld_total, 1), **{k: round(v, 1) for k, v in pld_cats.items()}},
    },
    "draft_target_split": sd_events_detail,
    "csys": {
        "csys_sd_ms_per_iter": round(csys_sd, 3),
        "csys_pld_ms_per_iter": round(csys_pld, 3),
    },
    "framework_amortization": {
        "Cd_standalone_ms": Cd_standalone,
        "Cd_inpipeline_ms": round(Cd_inpipeline, 2),
        "Cd_probe_ms": Cd_probe,
        "apparent_ratio": round(ratio, 1),
        "explanation": "measurement artifact from batched verify attribution, not real amortization",
    },
}

with open("profiler_results.json", "w") as f:
    json.dump(profiler_output, f, indent=2)

print("Saved profiler_results.json")
print(f"Final GPU: {torch.cuda.memory_allocated()/1e6:.0f} MB")

try:
    from google.colab import files
    files.download("profiler_results.json")
except:
    pass

Saved profiler_results.json
Final GPU: 8907 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>